# 6CS012 – Worksheet 6
## Practical Aspects of Training CNN for Image Classification

| | |
|---|---|
| **Module** | 6CS012 – Artificial Intelligence and Machine Learning |
| **Dataset** | FruitsInAmazon (6 classes) |
| **Runtime** | GPU recommended → Runtime → Change runtime type → T4 GPU |

### Contents
1. Setup & Imports
2. Mount Google Drive & Configure Paths
3. Data Understanding & Visualisation
   - 3.1 Read Directory & Extract Class Names
   - 3.2 Detect & Remove Corrupted Images
   - 3.3 Check Class Balance
   - 3.4 Visualise One Random Image per Class
4. Data Generation & Pre-processing
   - 4.1 Create Train / Validation Pipelines
   - 4.2 Visualise a Raw Batch
   - 4.3 Data Augmentation — Old API (`ImageDataGenerator`)
   - 4.4 Data Augmentation — New API (`tf.keras.layers.Random*`)
   - 4.5 Visualise Augmented Images
   - 4.6 Data Pre-processing: Rescaling
   - 4.7 Prefetch Pipelines
5. Model Building — CNN with BatchNormalization + Dropout

---
## Section 1 — Setup & Imports

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, Activation
)

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version :", tf.__version__)
print("GPU devices        :", tf.config.list_physical_devices('GPU'))

---
## Section 2 — Mount Google Drive & Configure Paths

Upload your **FruitsInAmazon** dataset to Google Drive and update the `BASE_DIR` path below.

Expected folder structure:
```
FruitsInAmazon/
├── train/
│   ├── acai/
│   ├── cupuacu/
│   ├── graviola/
│   ├── guarana/
│   ├── pupunha/
│   └── tucuma/
└── test/
    └── (same sub-folder structure)
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted successfully.")

In [ ]:
# ── UPDATE THIS PATH ──────────────────────────────────────────────────────────
BASE_DIR  = "/content/drive/MyDrive/FruitsInAmazon"
# ─────────────────────────────────────────────────────────────────────────────

train_dir = os.path.join(BASE_DIR, "train")
test_dir  = os.path.join(BASE_DIR, "test")

IMAGE_SIZE  = (224, 224)   # height × width
BATCH_SIZE  = 32
NUM_CLASSES = 6

print("train_dir :", train_dir)
print("test_dir  :", test_dir)
assert os.path.isdir(train_dir), "train_dir not found — check your path!"
assert os.path.isdir(test_dir),  "test_dir not found  — check your path!"

---
## Section 3 — Data Understanding & Visualisation

Three preliminary checks before building any model:
- **Data Integrity** — detect and remove corrupted images
- **Balanced Dataset** — check class distribution to avoid bias
- **Correct Labelling** — visually verify images match their class folders

### 3.1 — Read Directory & Extract Class Names

In [ ]:
# Each sub-directory inside train_dir represents one class label
class_names = sorted(os.listdir(train_dir))

if not class_names:
    print("ERROR: No class directories found in the train folder!")
else:
    print(f"Found {len(class_names)} classes: {class_names}")

# Expected output:
# Found 6 classes: ['acai', 'cupuacu', 'graviola', 'guarana', 'pupunha', 'tucuma']

### 3.2 — Detect & Remove Corrupted Images

`Image.verify()` checks the file header and structure without fully decoding the image.  
Any file that raises `IOError` or `UnidentifiedImageError` is corrupted and removed.

In [ ]:
corrupted_images = []   # collect paths of corrupted files

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        images = os.listdir(class_path)
        for img_name in images:
            img_path = os.path.join(class_path, img_name)
            try:
                with Image.open(img_path) as img:
                    img.verify()   # verify image integrity
            except (IOError, UnidentifiedImageError):
                corrupted_images.append(img_path)

if corrupted_images:
    print(f"\n{len(corrupted_images)} corrupted image(s) found:")
    for path in corrupted_images:
        print("  ", path)
        os.remove(path)   # remove so training does not break
    print("All corrupted images removed.")
else:
    print("No corrupted images found.")

# Expected output:
# No corrupted images found.

### 3.3 — Check Class Balance

A balanced dataset means no single class dominates training.  
If one class has significantly more images, the model may become biased toward it.

In [ ]:
class_counts = {}

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        images = [
            img for img in os.listdir(class_path)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        class_counts[class_name] = len(images)

# Print distribution table
print("\nClass Distribution:")
print("=" * 45)
print(f"{'Class Name':<25}{'Valid Image Count':>15}")
print("=" * 45)
for cls, cnt in class_counts.items():
    print(f"{cls:<25}{cnt:>15}")
print("=" * 45)

# Expected output:
# acai                               15
# cupuacu                            15
# graviola                           15
# guarana                            15
# pupunha                            15
# tucuma                             15

### 3.4 — Visualise One Random Image per Class

Confirms images are loaded correctly and representative of their class labels.

In [ ]:
selected_images = []   # store one image path per class
selected_labels = []   # store corresponding class names

for class_name in class_names:
    class_path = os.path.join(train_dir, class_name)
    if os.path.isdir(class_path):
        images = [
            img for img in os.listdir(class_path)
            if img.lower().endswith(('.png', '.jpg', '.jpeg'))
        ]
        if images:
            selected_img = os.path.join(class_path, random.choice(images))
            selected_images.append(selected_img)
            selected_labels.append(class_name)

# Display in a 2-row grid
num_classes = len(selected_images)
cols = (num_classes + 1) // 2   # 3 columns for 6 classes
rows = 2

fig, axes = plt.subplots(rows, cols, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    if i < num_classes:
        img = mpimg.imread(selected_images[i])
        ax.imshow(img)
        ax.set_title(selected_labels[i], fontsize=11)
        ax.axis("off")
    else:
        ax.axis("off")   # hide empty subplots

plt.suptitle("One Random Sample per Class", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Section 4 — Data Generation & Pre-processing

### 4.1 — Create Train / Validation tf.data Pipelines

`image_dataset_from_directory` handles:
- Reading images from class sub-folders
- Resizing to `IMAGE_SIZE`
- Batching into `BATCH_SIZE` groups
- Splitting into 80% train / 20% validation automatically

In [ ]:
train_ds, val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="both",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

# train_ds and val_ds are tf.data.Dataset objects.
# They do not have a fixed shape — they yield batches when iterated.
print("Class names:", train_ds.class_names)

# Inspect one batch shape
for images, labels in train_ds.take(1):
    print("Images shape:", images.shape)   # (32, 224, 224, 3)
    print("Labels shape:", labels.shape)   # (32,)

### 4.2 — Visualise a Raw Batch from the Training Set

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):   # take one batch from dataset
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(images[i]).astype("uint8"))
        plt.title(train_ds.class_names[labels[i]], fontsize=10)
        plt.axis("off")
plt.suptitle("Raw Training Batch (before augmentation)", fontsize=13)
plt.tight_layout()
plt.show()

### 4.3 — Data Augmentation: Old API (`ImageDataGenerator`)

The original Keras way of augmentation. Shown here for reference — **not used in training**.  
The new API (Section 4.4) is preferred and is what we use in this worksheet.

In [ ]:
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.preprocessing import image as keras_image

# datagen = ImageDataGenerator(
#     rotation_range=30,      # rotate up to 30 degrees
#     width_shift_range=0.2,  # shift width by 20%
#     height_shift_range=0.2, # shift height by 20%
#     shear_range=0.2,        # shear transformation
#     zoom_range=0.2,         # zoom in/out by 20%
#     horizontal_flip=True,   # flip images horizontally
#     fill_mode='nearest'     # fill missing pixels after transformation
# )

# # Load one sample image to demonstrate augmentation
# sample_img_path = selected_images[0]   # reuse the random image from Section 3.4
# img = keras_image.load_img(sample_img_path, target_size=IMAGE_SIZE)
# x   = keras_image.img_to_array(img)     # convert to NumPy array
# x   = np.expand_dims(x, axis=0)         # shape: (1, 224, 224, 3)

# # Generate and display 7 augmented versions
# aug_iter = datagen.flow(x, batch_size=1)
# fig, ax  = plt.subplots(1, 7, figsize=(15, 5))
# for i in range(7):
#     batch = next(aug_iter)
#     ax[i].imshow(batch[0].astype('uint8'))
#     ax[i].axis('off')
# plt.suptitle("ImageDataGenerator — 7 Augmented Versions (Old API)", fontsize=12)
# plt.tight_layout()
# plt.show()

### 4.4 — Data Augmentation: New API (`tf.keras.layers.Random*`)

**This is the recommended approach for this worksheet and your portfolio project.**

| Feature | `ImageDataGenerator` (old) | `tf.keras.layers.Random*` (new) |
|---|---|---|
| Runs on | CPU only | GPU (part of the model graph) |
| Active at test time | Yes (must be disabled manually) | No — auto-disabled during `evaluate()` / `predict()` |
| Integration | External to model | Embedded inside model |

These layers will be embedded directly into the model in Section 5.

In [ ]:
# Define augmentation layers — will be embedded inside the model
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),     # horizontal mirror
    layers.RandomRotation(0.1),          # ±10% of 360°
]

def data_augmentation(images):
    """Apply all augmentation layers sequentially to a batch."""
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

# NOTE: You can add more augmentation layers for your project.
# Check Keras docs: https://keras.io/api/layers/preprocessing_layers/
# e.g. layers.RandomZoom, layers.RandomTranslation, layers.RandomContrast

print("Augmentation layers defined:")
for l in data_augmentation_layers:
    print(f"  {l.__class__.__name__}")

### 4.5 — Visualise Augmented Images

Applying `data_augmentation` 9 times to the same image shows the variety of transformations the model will see during training.

In [ ]:
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)           # new random transform each call
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(np.array(augmented_images[0]).astype("uint8"))  # show first image each time
        plt.axis("off")
plt.suptitle("9 Different Augmentations of the Same Image (New API)", fontsize=13)
plt.tight_layout()
plt.show()

### 4.6 — Data Pre-processing: Rescaling

RGB pixel values are in `[0, 255]`. Neural networks train better with small input values,  
so we rescale to `[0, 1]` using a `Rescaling(1./255)` layer.

**Two approaches — we use Option 2 (recommended):**

**Option 1 — Apply to dataset externally:**
```python
augmented_train_ds = train_ds.map(
    lambda x, y: (data_augmentation(x, training=True), y)
)
```
Runs on CPU, augmentation happens before rescaling outside the model.

**Option 2 — Embed inside the model (recommended):**
```python
inputs = keras.Input(shape=input_shape)
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)
# ... rest of model
```
Runs on GPU during `fit()`. Augmentation is auto-disabled at `evaluate()`/`predict()` time.  
Rescaling is always applied — including at inference — because it is part of the model.

### 4.7 — Prefetch Pipelines for Performance

`prefetch()` overlaps CPU data loading with GPU model computation,  
eliminating idle time between batches.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# cache()    → stores dataset in RAM after first epoch (fast for small datasets)
# prefetch() → overlaps I/O and GPU compute
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Pipelines optimised with cache + prefetch. ✓")

---
## Section 5 — Model Building
### CNN with BatchNormalization + Dropout

#### BatchNormalization — how it works

Normalises the pre-activations of each layer using the batch mean and variance:

$$Z' = \frac{Z - \mu_B}{\sqrt{\sigma^2_B + \epsilon}}$$

Then scales and shifts by learnable parameters γ and β:

$$a' = \gamma Z' + \beta$$

**Effect:** Reduces internal covariate shift → faster convergence, more stable training.

#### Dropout — how it works

Randomly sets a fraction `r` of neurons to zero during each forward pass in training:

$$\hat{X} = \text{Dropout}(X, r)$$

At inference, all neurons are used but outputs are scaled by $\frac{1}{1-r}$ to preserve expected values:

$$Y = \frac{\hat{X}}{1 - r}$$

**Effect:** Prevents co-adaptation of neurons → reduces overfitting.

#### Key design decisions

| Decision | Reason |
|---|---|
| `activation=None` in Conv2D/Dense | BN is applied to pre-activations; ReLU follows after |
| BN before Activation | Standard ordering from original BN paper |
| Dropout(0.25) after pooling | Light regularisation in conv blocks |
| Dropout(0.5) in FC layers | Stronger regularisation where overfitting risk is highest |
| Rescaling inside model | Applied consistently at both train and inference time |
| Augmentation inside model | GPU-accelerated; auto-disabled at eval/predict |

#### Architecture
```
Input (224, 224, 3)
 └─ Lambda [Augmentation]    ← active only during fit()
 └─ Rescaling (÷255)         ← always active
 └─ Conv Block 1: Conv2D(32)  → BN → ReLU → MaxPool(2,2) → Dropout(0.25)
 └─ Conv Block 2: Conv2D(64)  → BN → ReLU → MaxPool(2,2) → Dropout(0.25)
 └─ Conv Block 3: Conv2D(128) → BN → ReLU → MaxPool(2,2) → Dropout(0.25)
 └─ Conv Block 4: Conv2D(256) → BN → ReLU → MaxPool(2,2) → Dropout(0.25)
 └─ Flatten
 └─ FC Block 1: Dense(512) → BN → ReLU → Dropout(0.5)
 └─ FC Block 2: Dense(256) → BN → ReLU → Dropout(0.5)
 └─ FC Block 3: Dense(128) → BN → ReLU → Dropout(0.5)
 └─ FC Block 4: Dense(64)  → BN → ReLU → Dropout(0.5)
 └─ Output: Dense(6, softmax)
```

In [ ]:
input_shape = IMAGE_SIZE + (3,)   # (224, 224, 3)

model = Sequential([

    layers.Input(shape=input_shape),

    # ── Augmentation + Rescaling ─────────────────────────────────────────────
    layers.Lambda(data_augmentation),   # GPU-accelerated; disabled at eval/predict
    layers.Rescaling(1.0 / 255),        # [0,255] → [0,1]

    # ── Convolutional Block 1 ────────────────────────────────────────────────
    # padding='same' keeps spatial dims after convolution
    # activation=None → BN then Activation added separately
    Conv2D(32, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),               # 224 → 112
    Dropout(0.25),

    # ── Convolutional Block 2 ────────────────────────────────────────────────
    Conv2D(64, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),               # 112 → 56
    Dropout(0.25),

    # ── Convolutional Block 3 ────────────────────────────────────────────────
    Conv2D(128, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),               # 56 → 28
    Dropout(0.25),

    # ── Convolutional Block 4 ────────────────────────────────────────────────
    Conv2D(256, (3, 3), padding='same', activation=None),
    BatchNormalization(),
    Activation('relu'),
    MaxPooling2D((2, 2)),               # 28 → 14
    Dropout(0.25),

    # ── Flatten ──────────────────────────────────────────────────────────────
    # Converts (14, 14, 256) feature maps → 1D vector of 50,176 values
    Flatten(),

    # ── Fully Connected Block 1 ──────────────────────────────────────────────
    Dense(512, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    # ── Fully Connected Block 2 ──────────────────────────────────────────────
    Dense(256, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    # ── Fully Connected Block 3 ──────────────────────────────────────────────
    Dense(128, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    # ── Fully Connected Block 4 ──────────────────────────────────────────────
    Dense(64, activation=None),
    BatchNormalization(),
    Activation('relu'),
    Dropout(0.5),

    # ── Output Layer ─────────────────────────────────────────────────────────
    # 6 neurons = 6 fruit classes
    # softmax converts raw logits into a probability distribution summing to 1
    Dense(NUM_CLASSES, activation='softmax')

], name="FruitsInAmazon_CNN_BN_Dropout")

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',   # integer labels → sparse variant
    metrics=['accuracy']
)

model.summary()